# Lab 10: Few-Shot Structured Output

## Showing, not just telling

The next cells build `build_few_shot_messages(examples, query)`, which turns a list of worked `(review, result_dict)` pairs into prior user/assistant turns placed before the real question, and `parse_json_response(text)`, which turns the model's text reply back into a Python dict. See the Reading tab for why worked examples steer format better than instructions alone, and how JSON parsing/validation works.

### Setup - get a free key at https://console.groq.com/keys

In [ ]:
"""Get your free Groq API key at https://console.groq.com/keys - it's never written to this notebook."""
import getpass
import json
import requests

GROQ_API_URL = "https://api.groq.com/openai/v1/chat/completions"
DEFAULT_MODEL = "llama-3.1-8b-instant"

GROQ_API_KEY = getpass.getpass("Enter your GROQ_API_KEY: ")


def build_request_payload(messages, model=DEFAULT_MODEL, temperature=0.7, response_format=None):
    """The request BODY: what we're asking the model to do."""
    payload = {"model": model, "messages": messages, "temperature": temperature}
    if response_format is not None:
        payload["response_format"] = response_format
    return payload


def extract_reply(response_json):
    """Pulls the assistant's text out of the RESPONSE body."""
    return response_json["choices"][0]["message"]["content"]


def call_groq(messages, api_key=None, model=DEFAULT_MODEL, temperature=0.7, response_format=None):
    """The full request: ENDPOINT + METHOD (POST) + HEADERS (auth) + BODY -> RESPONSE."""
    headers = {
        "Authorization": f"Bearer {api_key or GROQ_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = build_request_payload(messages, model=model, temperature=temperature, response_format=response_format)
    response = requests.post(GROQ_API_URL, headers=headers, json=payload, timeout=30)
    response.raise_for_status()
    return extract_reply(response.json())


print("Groq setup ready. Call call_groq(messages) to send a chat completion request.")


### Concept 1 practice: teaching by example, not by rule
A quick runnable version of "show examples, let the pattern speak for itself" before you build it into a prompt.

In [ ]:
examples = [
    ("hello", "HELLO!"),
    ("bye", "BYE!"),
]
# Instead of describing the rule in words, just show the pattern:
def apply_pattern(word):
    return word.upper() + "!"

query = "yes"
print(apply_pattern(query))  # expected output: YES!

### Concept 2 practice: parsing (and trusting) JSON output
A small parse-then-validate example on a different bit of JSON, using the same try/except shape you'll need for a model's reply.

In [ ]:
import json

raw = '{"item": "Mug", "price": 9.99}'

try:
    product = json.loads(raw)
    if "price" not in product:
        raise ValueError("missing price")
    print(product["item"], product["price"])  # expected output: Mug 9.99
except json.JSONDecodeError:
    print("not valid JSON")
except ValueError as e:
    print(f"parsed but incomplete: {e}")

## Exercise

Implement `build_few_shot_messages(examples, query)` - `examples` is a list of `(review, result_dict)` pairs shown as prior user/assistant turns - and `parse_json_response(text)`, which has to cope with models that wrap JSON in ```json fences even when told not to.

In [ ]:
import json


def build_few_shot_messages(examples: list[tuple[str, dict]], query: str) -> list[dict]:
    """TODO: build a messages list: one system message instructing the model
    to reply with ONLY a JSON object of the shape {"sentiment": ..., "confidence": ...},
    then one user/assistant turn per example (user=review text, assistant=
    json.dumps(result_dict)), then a final user message with `query`."""
    raise NotImplementedError


def parse_json_response(text: str) -> dict:
    """TODO: strip a leading ```json / ``` fence and trailing ``` if present,
    then json.loads the remainder and return it. Raise ValueError if the
    result is missing the 'sentiment' or 'confidence' keys."""
    raise NotImplementedError


examples = [("Amazing!", {"sentiment": "positive", "confidence": 0.95})]
messages = build_few_shot_messages(examples, "Terrible.")
assert messages[0]["role"] == "system"
assert messages[-1] == {"role": "user", "content": "Terrible."}
assert any(m["role"] == "assistant" for m in messages)

parsed = parse_json_response('```json\n{"sentiment": "negative", "confidence": 0.8}\n```')
assert parsed == {"sentiment": "negative", "confidence": 0.8}

try:
    parse_json_response('{"sentiment": "negative"}')
    raise AssertionError("should have raised ValueError for a missing key")
except ValueError:
    pass

print("All checks passed.")

## Solution

In [ ]:
def build_few_shot_messages(examples: list[tuple[str, dict]], query: str) -> list[dict]:
    messages = [
        {
            "role": "system",
            "content": "Classify review sentiment. Reply with ONLY a JSON object of the shape "
                       '{"sentiment": "positive" | "negative", "confidence": 0.0-1.0}. No other text.',
        }
    ]
    for review, result in examples:
        messages.append({"role": "user", "content": review})
        messages.append({"role": "assistant", "content": json.dumps(result)})
    messages.append({"role": "user", "content": query})
    return messages


def parse_json_response(text: str) -> dict:
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`")
        cleaned = cleaned.removeprefix("json").strip()
    result = json.loads(cleaned)
    missing = {"sentiment", "confidence"} - result.keys()
    if missing:
        raise ValueError(f"Missing required key(s): {sorted(missing)}")
    return result


print(build_few_shot_messages(examples, "Terrible."))

## Real-world project (LIVE - needs your GROQ_API_KEY above)

Asks for structured `{"sentiment": ..., "confidence": ...}` JSON on real sentences, validates the parse, and tabulates results.

In [ ]:
"""Load the shared real-world review dataset (embedded so this notebook needs no repo checkout)."""
import csv
import io

REVIEW_CSV = """sentence,label
the movie was absolutely wonderful and kept me hooked the whole time,1
i loved every minute of this film,1
the acting was superb and the story was touching,1
this was one of the best movies i have seen all year,1
the plot was engaging and the pacing was perfect,1
a brilliant film with outstanding performances,1
the soundtrack was beautiful and matched the mood perfectly,1
i would watch this movie again without hesitation,1
the director did an amazing job telling this story,1
this film exceeded all of my expectations,1
the characters felt real and the dialogue was sharp,1
a truly moving and unforgettable experience,1
the visuals were stunning and the effects were impressive,1
this is a masterpiece that everyone should see,1
the humor was clever and the timing was excellent,1
i was on the edge of my seat the entire film,1
the ending was satisfying and well earned,1
this movie made me laugh and cry at the same time,1
a wonderfully crafted story with great performances,1
the chemistry between the leads was fantastic,1
i really enjoyed the twist near the end,1
this film is a delightful surprise from start to finish,1
the cinematography was gorgeous throughout the movie,1
a heartwarming tale that left me smiling,1
the pacing kept the tension high without ever dragging,1
i cannot recommend this movie enough,1
the product works exactly as advertised and arrived quickly,1
i am very happy with this purchase,1
the quality is excellent for the price,1
this item exceeded my expectations in every way,1
the customer service team was friendly and helpful,1
setup was quick and the instructions were clear,1
this gadget has made my daily routine so much easier,1
the build quality feels sturdy and well made,1
i would definitely buy this product again,1
the battery life is impressive and lasts all day,1
shipping was fast and the packaging was excellent,1
this is by far the best purchase i made this year,1
the design is sleek and it works flawlessly,1
i appreciate how easy this product is to use,1
the price is fair given the quality you get,1
this laptop is fast reliable and great for work,1
the headphones sound amazing and are comfortable to wear,1
i love how lightweight and portable this device is,1
the app is intuitive and runs smoothly on my phone,1
this coffee maker brews a delicious cup every morning,1
the movie was boring and dragged on for way too long,0
i did not enjoy this film at all,0
the acting felt wooden and the story made no sense,0
this was one of the worst movies i have seen all year,0
the plot was confusing and the pacing was terrible,0
a disappointing film with weak performances,0
the soundtrack was forgettable and did not fit the mood,0
i would never watch this movie again,0
the director seemed to have no clear vision for this story,0
this film fell far short of my expectations,0
the characters felt flat and the dialogue was awkward,0
a frustrating and forgettable experience overall,0
the visuals were dull and the effects looked cheap,0
this movie is a mess that nobody should waste time on,0
the humor fell flat and the timing was off,0
i kept checking the clock throughout the entire film,0
the ending felt rushed and unearned,0
this movie made me want to leave the theater early,0
a poorly crafted story with unconvincing performances,0
the chemistry between the leads felt forced,0
i was annoyed by the predictable twist near the end,0
this film is a tedious slog from start to finish,0
the cinematography was flat and uninspired throughout,0
a dull tale that left me checking my phone,0
the pacing dragged and the tension never built,0
i cannot recommend this movie to anyone,0
the product broke after only two days of use,0
i am very unhappy with this purchase,0
the quality is poor for the price charged,0
this item fell short of my expectations in every way,0
the customer service team was rude and unhelpful,0
setup was confusing and the instructions were unclear,0
this gadget has made my daily routine more frustrating,0
the build quality feels flimsy and cheaply made,0
i would never buy this product again,0
the battery life is disappointing and drains quickly,0
shipping was slow and the packaging was damaged,0
this is by far the worst purchase i made this year,0
the design is clunky and it barely works,0
i am frustrated by how hard this product is to use,0
the price is too high given the poor quality,0
this laptop is slow unreliable and bad for work,0
the headphones sound tinny and are uncomfortable to wear,0
i hate how heavy and bulky this device is,0
the app is confusing and crashes on my phone,0
this coffee maker leaks and ruins my morning,0
the sequel was even better than the original film,1
i found the story dull and the characters uninteresting,0
the special effects were groundbreaking and beautifully done,1
the remote control stopped working within a week,0
this phone case fits perfectly and looks great,1
the instructions were missing several important steps,0
the keyboard feels responsive and is a joy to type on,1
the screen arrived scratched and the seller ignored my complaint,0
this backpack is durable and has plenty of storage,1
the movie relied on tired clichés and predictable jokes,0
the restaurant served delicious food with generous portions,1
our waiter was attentive and the service was outstanding,1
the pasta was cooked perfectly and full of flavor,1
this cafe has a cozy atmosphere and friendly staff,1
the steak was tender and cooked exactly how i asked,1
dessert was the highlight of an already fantastic meal,1
the wine pairing recommendations were spot on,1
i loved the fresh ingredients and creative menu,1
the bakery bread is soft and always baked fresh,1
this bistro has become my favorite spot in town,1
the book kept me turning pages late into the night,1
the author writes with wit and genuine emotional depth,1
i finished this novel in a single weekend it was that good,1
the world building in this book is rich and immersive,1
this memoir was honest funny and deeply moving,1
the mystery had twists i never saw coming,1
the poetry collection was beautiful and thought provoking,1
i recommend this book to anyone who loves a good story,1
the hotel room was spotless and beautifully decorated,1
the staff greeted us warmly and upgraded our room,1
the pool area was relaxing and well maintained,1
check in was smooth and the concierge was helpful,1
the breakfast buffet had a wonderful variety of fresh food,1
our stay at this resort exceeded every expectation,1
the view from the balcony was absolutely breathtaking,1
the gym and spa facilities were top notch,1
this hotel is perfect for a relaxing weekend getaway,1
the car handles smoothly and the ride is comfortable,1
fuel efficiency on this vehicle has been impressive,1
the dealership made the buying process quick and painless,1
this vacuum cleaner picks up dirt with ease,1
the blender is powerful and easy to clean,1
these shoes are comfortable and have held up well,1
the jacket is warm stylish and fits perfectly,1
the restaurant food was bland and arrived cold,0
our waiter seemed annoyed and service was painfully slow,0
the pasta was overcooked and lacked any real flavor,0
this cafe was noisy and the staff were dismissive,0
the steak was tough and clearly overdone,0
dessert was stale and ruined an otherwise average meal,0
the wine pairing recommendations were completely off,0
i disliked the stale ingredients and boring menu,0
the bakery bread is dry and often stale,0
this bistro has become a place i actively avoid,0
the book dragged and i never finished reading it,0
the author writes with clumsy prose and shallow characters,0
i abandoned this novel halfway through out of boredom,0
the world building in this book felt lazy and inconsistent,0
this memoir was dull repetitive and emotionally flat,0
the mystery was predictable from the very first page,0
the poetry collection felt uninspired and forgettable,0
i would not recommend this book to anyone,0
the hotel room was dirty and poorly maintained,0
the staff ignored us and refused to help with our request,0
the pool area was filthy and badly maintained,0
check in took forever and the concierge was unhelpful,0
the breakfast buffet had stale food and few options,0
our stay at this resort fell far below expectations,0
the view from the balcony was blocked by construction,0
the gym and spa facilities were outdated and dirty,0
this hotel is a poor choice for any kind of getaway,0
the car rattles constantly and the ride is uncomfortable,0
fuel efficiency on this vehicle has been disappointing,0
the dealership made the buying process stressful and confusing,0
this vacuum cleaner barely picks up any dirt,0
the blender is weak and difficult to clean,0
these shoes are uncomfortable and fell apart quickly,0
the jacket is thin flimsy and does not fit well,0
"""

def load_reviews():
    reader = csv.DictReader(io.StringIO(REVIEW_CSV))
    sentences, labels = [], []
    for row in reader:
        sentences.append(row["sentence"])
        labels.append(int(row["label"]))
    return sentences, labels

sentences, labels = load_reviews()
print(f"Loaded {len(sentences)} real review sentences.")


In [ ]:
FEW_SHOT_EXAMPLES = [
    ("This is the best purchase I've made all year.", {"sentiment": "positive", "confidence": 0.97}),
    ("It broke after two days and support ignored me.", {"sentiment": "negative", "confidence": 0.95}),
]

N_SAMPLES = 8
results = []
for review in sentences[:N_SAMPLES]:
    reply = call_groq(
        build_few_shot_messages(FEW_SHOT_EXAMPLES, review),
        response_format={"type": "json_object"},
    )
    try:
        parsed = parse_json_response(reply)
        results.append((review, parsed["sentiment"], parsed["confidence"]))
    except (json.JSONDecodeError, ValueError) as e:
        results.append((review, f"PARSE ERROR: {e}", None))

for review, sentiment, confidence in results:
    print(f"{sentiment!s:<25} conf={confidence}  {review[:55]}")